# DSS v1.1 — Résumé exécutif, urgence, décisions métier

## Objectif

La v1.0 explique le risque et chiffre l'impact. La v1.1 rend le prototype présentable en réunion de direction :

- **Résumé exécutif** en tête de page — la situation nationale en 15 secondes
- **Causes enrichies** — y compris l'absence de redistribution possible
- **Horizon temporel** — combien de jours de stock reste-t-il, au rythme actuel ?
- **Priorités nationales** — où concentrer les ressources en premier
- **Stratégies notées** — impact / effort en étoiles, avec une recommandation explicite
- **Scénarios nommés** — le décideur choisit une décision métier, pas des curseurs techniques

Toujours des règles arithmétiques transparentes. L'estimation "jours de stock restants" est un rythme de consommation, pas une prévision — elle est présentée comme telle partout dans le prototype.

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append(str(Path.cwd()))
from src import decision_engine as de

df = pd.read_csv(Path.cwd() / "cnts_simulated_blood_demand.csv")
latest_month = df["month"].max()

snapshot, transfers = de.build_recommendations(df, latest_month)
snapshot = de.explain_causes(snapshot, df, latest_month, transfers=transfers)
latest_month

'2025-12'

## 1. Résumé exécutif

Ce que verrait le Directeur Général en premier, avant tout tableau.

In [2]:
summary = de.build_executive_summary(snapshot, transfers)
print(f"Situation nationale — {latest_month}\n")
for line in de.executive_summary_text(summary):
    print("-", line)

Situation nationale — 2025-12

- 5 region(s) presentent un risque eleve.
- Le deficit estime est de 605 unites.
- Les groupes O+ et B+ sont les plus sous tension.
- La priorite est une campagne ciblee a Bouaké et Abidjan.
- Aucune redistribution suffisante n'est disponible ce mois-ci : la priorite est la collecte.


## 2. Priorités nationales — où agir en premier

Classement par urgence réelle (jours de stock restants), pas seulement par niveau de risque — deux régions "Élevé" n'ont pas forcément le même délai avant rupture.

In [3]:
de.build_priority_list(snapshot, top_n=5)

,region,blood_group,risk_level_calc,jours_de_stock
0,San Pedro,O-,Eleve,12.4
1,Korhogo,O-,Eleve,14.1
2,Yamoussoukro,B-,Eleve,15.0
3,San Pedro,B-,Eleve,15.0
4,Bouaké,O-,Eleve,15.5


## 3. Causes enrichies

La situation la plus urgente de la liste ci-dessus, avec toutes ses causes — y compris l'absence de redistribution disponible quand c'est le cas.

In [4]:
top = snapshot.iloc[0]
print(f"{top['region']} / {top['blood_group']} — risque {top['risk_level_calc']}, "
      f"{de.estimate_days_of_stock(top)} jours de stock restants\n")
for cause in top["causes"]:
    print("-", cause)

Bouaké / O+ — risque Eleve, 20.5 jours de stock restants

- Hausse inhabituelle de la demande : 243 unites contre 191 en moyenne sur les 3 derniers mois (27% de plus).
- Stock sous le seuil de securite : couverture de 0.68 contre 0.85 minimum recommande.
- Aucune region excedentaire disponible pour redistribuer ce groupe sanguin ce mois-ci.


## 4. Scénarios nommés — le décideur choisit une décision, pas des unités

Le simulateur ne demande plus "combien d'unités ?" mais "quelle décision ?". Chaque option est déjà chiffrée par le moteur.

In [5]:
for label, params in de.available_scenarios(top, transfers).items():
    scenario = de.simulate_scenario(top, **params)
    print(f"« {label} »")
    print(f"   -> risque {scenario['risk_level_scenario']} (couverture {scenario['coverage_ratio_scenario']:.2f})\n")

« Organiser une campagne ciblee a Bouaké (+43 unites) »
   -> risque Moyen (couverture 0.86)

« Ne rien faire »
   -> risque Eleve (couverture 0.68)



## 5. Stratégies notées, avec recommandation

Sur l'exemple de juin 2024 (Bouaké / AB+, transfert disponible) : le moteur compare et recommande explicitement l'option la moins coûteuse pour un résultat au moins équivalent.

In [6]:
snap_june, transfers_june = de.build_recommendations(df, "2024-06")
row_bouake_abplus = snap_june[(snap_june["region"] == "Bouaké") & (snap_june["blood_group"] == "AB+")].iloc[0]

de.compare_strategies(row_bouake_abplus, transfers_june)

,strategie,detail,risque_resultant,impact,effort,recommandee
0,Transfert,5 unites depuis Abidjan,Moyen,★★★☆☆,★★☆☆☆,✅ Recommandee
1,Campagne de collecte,+1 unites collectees,Moyen,★★★☆☆,★★★☆☆,
2,Aucune action,Statu quo,Eleve,★☆☆☆☆,☆☆☆☆☆,


## Ce qui est délibérément hors scope de cette passe

- **Scénarios multi-régions** ("campagne nationale", "renforcer partout") : demanderaient une section dédiée plutôt que d'être compressés dans le panneau par situation. Prochaine itération.
- **Prévision à 30 jours** : reste en v2.0, nécessite des données réelles pour être calibrée sérieusement.

## Ce que ça démontre

Le prototype répond maintenant à 6 questions de décideur : *quoi surveiller, pourquoi, dans combien de temps, quel impact, et si, et face à quelle alternative* — sans une seule ligne de modèle prédictif.